In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/

/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed


In [3]:
!pip install -q datasets

# 라이브러리

In [4]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
import copy

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# Torch 모델 구조
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# BERT Model
# Fine Tuning
from transformers import AutoModel, AutoTokenizer
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers.modeling_outputs import SequenceClassifierOutput


# sklearn 전처리, 데이터 분할, 성능 평가지표
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils

In [5]:
class ReviewPairDataset(Dataset):
    def __init__(self, user_texts, item_texts, user_ids, item_ids, ratings, tokenizer, max_length=512):
        self.user_texts = user_texts
        self.item_texts = item_texts
        self.user_ids = user_ids
        self.item_ids = item_ids
        self.ratings = ratings
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.user_texts)

    def __getitem__(self, idx):
        user_enc = self.tokenizer(self.user_texts[idx], truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt")
        item_enc = self.tokenizer(self.item_texts[idx], truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt")
        user_id = torch.tensor(self.user_ids[idx], dtype=torch.long)
        item_id = torch.tensor(self.item_ids[idx], dtype=torch.long)
        rating = torch.tensor(float(self.ratings[idx]), dtype=torch.float)

        return {
            "input_ids_user": user_enc["input_ids"].squeeze(0),
            "attention_mask_user": user_enc["attention_mask"].squeeze(0),
            "input_ids_item": item_enc["input_ids"].squeeze(0),
            "attention_mask_item": item_enc["attention_mask"].squeeze(0),
            "labels": rating,
            "user_id": user_id,
            "item_id": item_id
        }

In [6]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions  # 그대로 사용!

    # 지표 계산
    mae = mean_absolute_error(labels, preds)
    mse = mean_squared_error(labels, preds)
    rmse = np.sqrt(mse)

    epsilon = 1e-8
    mape = np.mean(np.abs((labels - preds) / (labels + epsilon))) * 100

    return {
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "mape": mape
    }

In [7]:
class Co_Attention_Layer(nn.Module):
    def __init__(self, emb_dim=768, nhead=8):
        super().__init__()
        self.attn_user_to_item = nn.MultiheadAttention(embed_dim=emb_dim, num_heads=nhead, batch_first=True)
        self.attn_item_to_user = nn.MultiheadAttention(embed_dim=emb_dim, num_heads=nhead, batch_first=True)

    def forward(self, user_vec, item_vec):
        # 입력: [batch, emb]
        user_vec = user_vec.unsqueeze(1)  # [batch, 1, emb]
        item_vec = item_vec.unsqueeze(1)  # [batch, 1, emb]

        # ① user attends to item (Q=user, K/V=item)
        attn_u2i, _ = self.attn_user_to_item(query=user_vec, key=item_vec, value=item_vec)  # [batch, 1, emb]

        # ② item attends to user (Q=item, K/V=user)
        attn_i2u, _ = self.attn_item_to_user(query=item_vec, key=user_vec, value=user_vec)  # [batch, 1, emb]

        # Flatten and concat both directions
        concat = torch.cat([attn_u2i.squeeze(1), attn_i2u.squeeze(1)], dim=-1)  # [batch, emb*2]
        return concat

# 데이터셋 로딩

In [8]:
# 데이터 로딩
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_segment_v5_final.pkl')
print(df.shape)
df.head()

(66718, 11)


,user_id,asin,parent_asin,text,rating,title_y,small_image_url,medium_image_url,large_image_url,LLaMA_Output,LLaVA_Output
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"Based on this review, the user appears to pref...",The Udalyn 6pairs Stud Earrings Set is designe...
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08JQ88186,B08JQ88186,Great little earrings. Perfect size. Not too b...,5,ZENI 925 Sterling Silver Snowflake Earrings fo...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"Based on this review, the user appears to pref...","The product, a pair of sterling silver snowfla..."
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"Based on this review, the user appears to pref...","This product, a bangle key ring bracelet, is d..."
3,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07RHSJQQN,B07RHSJQQN,This is a nice patriotic tank. The medium fits...,4,YEXIPO Womens American Flag Racerback Tank Top...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"Based on this review, the user appears to pref...","This product, a women's American flag racerbac..."
4,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07LFZRYC2,B07LFZRYC2,Great boys shirt. My grandson wears a medium i...,5,Hurley Boys' One and Ony Graphic T-Shirt,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"Based on this review, the user appears to pref...",The Hurley Boys' One and Ony Graphic T-Shirt i...


# Prepare Input

In [9]:
# Encode user and item IDs
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

# fit-transform
df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

# 실험 데이터의 고유한 user와 item 수를 집계
n_users = df['user_id'].nunique()
n_items = df['asin'].nunique()

Num_User = n_users + 1
Num_Item = n_items + 1

In [10]:
# train, valid, test으로 데이터셋 분리
train, test = train_test_split(df, test_size=0.2, random_state=42)
train, valid = train_test_split(train, test_size=0.125, random_state=42)

train = train.reset_index(drop=True)
valid = valid.reset_index(drop=True)
test = test.reset_index(drop=True)

In [11]:
train.shape, valid.shape, test.shape

((46702, 11), (6672, 11), (13344, 11))

In [12]:
n_users, n_items

(7449, 16099)

In [13]:
# train, test, valid 변수 선언
train_user_id = train['user_id'].to_numpy()
train_item_id = train['asin'].to_numpy()
train_summary = train['LLaMA_Output']
train_image_description = train['LLaVA_Output']
train_rating = train['rating'].to_numpy()

valid_user_id = valid['user_id'].to_numpy()
valid_item_id = valid['asin'].to_numpy()
valid_summary = valid['LLaMA_Output']
valid_image_description = valid['LLaVA_Output']
valid_rating = valid['rating'].to_numpy()

test_user_id = test['user_id'].to_numpy()
test_item_id = test['asin'].to_numpy()
test_summary = test['LLaMA_Output']
test_image_description = test['LLaVA_Output']
test_rating = test['rating'].to_numpy()

# Load Model

In [14]:
# BERT 모델 Loading
# 해당 모델은 Layer 6개인 모델

# 사용자 텍스트 처리 모델
BERT_review = AutoModel.from_pretrained("bert-base-uncased")

# 아이템 텍스트 처리 모델
BERT_image = AutoModel.from_pretrained("bert-base-uncased")

# tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

# Two Tower BERT LoRA

```python
MAS_Rec(
  user_id: Embedding(num_users, 64)
  item_id: Embedding(num_items, 64)
  user_model: BertModel
  item_model: BertModel
  cls_dense: MLP(768→512→256)
  co_attn: Co_Attention_Layer(MultiheadAttention*2, emb_dim=320)
  fc_final: MLP(640→256→1)
)

```

In [15]:
def make_mlp(sizes, activation=nn.ReLU):
    """
    중간 feature extractor용 MLP
    모든 Linear 뒤에 activation 함수가 붙음 (마지막 레이어 포함)
    """
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i+1]))
        layers.append(activation())
    return nn.Sequential(*layers)

In [16]:
def make_last_mlp(sizes, activation=nn.ReLU):
    """
    최종 예측 MLP
    마지막 Linear 뒤에는 activation이 없음 (즉, 마지막은 Linear-only)
    """
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i+1]))
        if i < len(sizes) - 2:
            layers.append(activation())
    return nn.Sequential(*layers)

In [17]:
class MAS_Rec(nn.Module):
    def __init__(self, n_users, n_items, user_model, item_model, nhead=8):
        super().__init__()

        # Interaction IDs
        self.user_id = nn.Embedding(n_users, 64)
        self.item_id = nn.Embedding(n_items, 64)

        # Two Tower Bert
        self.user_model = user_model
        self.item_model = item_model

        # [CLS] Dense
        self.cls_dense = make_mlp([768, 512, 256])

        # Co-Attention (bi-directional cross attention)
        # 256([CLS] dense output emb) + 64(interaction emb)
        self.cross_attn = Co_Attention_Layer(emb_dim=320, nhead=nhead)

        # Last Dense
        self.fc_final = make_last_mlp([640, 256, 1])

    def forward(self, user_id, item_id, input_ids_user, attention_mask_user, input_ids_item, attention_mask_item, labels=None):

        # Interaction IDs
        user_emb = self.user_id(user_id)    # [batch, 64]
        item_emb = self.item_id(item_id)    # [batch, 64]

        # Two Tower BERT
        # 여기에서 input_ids는 token_id(token index)
        user_cls = self.user_model(input_ids=input_ids_user, attention_mask=attention_mask_user).last_hidden_state[:, 0, :]
        item_cls = self.item_model(input_ids=input_ids_item, attention_mask=attention_mask_item).last_hidden_state[:, 0, :]

        # [CLS] Dense
        user_cls = self.cls_dense(user_cls)
        item_cls = self.cls_dense(item_cls)

        # Concat
        user_feat = torch.cat([user_emb, user_cls], dim=-1)
        item_feat = torch.cat([item_emb, item_cls], dim=-1)

        # Co-Attetnion
        co_attn_output = self.cross_attn(user_feat, item_feat)  # [batch, 512]

        # Last Dense Layer
        logits = self.fc_final(co_attn_output).squeeze(1)

        if labels is not None:
            loss = nn.MSELoss()(logits, labels)
            return {"loss": loss, "logits": logits}
        return {"logits": logits}

# Fine Tuning

In [18]:
# 모델 생성
model = MAS_Rec(Num_User, Num_Item, user_model=BERT_review, item_model=BERT_image, nhead=8)

In [19]:
for name, module in model.named_children():
    print(name, module)

user_id Embedding(7450, 64)
item_id Embedding(16100, 64)
user_model BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementw

In [20]:
# 실제 훈련용
train_dataset = ReviewPairDataset(train_summary, train_image_description, train_user_id, train_item_id, train_rating, tokenizer, max_length=512)
valid_dataset = ReviewPairDataset(valid_summary, valid_image_description, valid_user_id, valid_item_id, valid_rating, tokenizer, max_length=512)
test_dataset = ReviewPairDataset(test_summary, test_image_description, test_user_id, test_item_id, test_rating, tokenizer, max_length=512)

In [21]:
# 하이퍼파라미터
num_training_samples = 46702 # 16443 # 46702
per_device_train_batch_size = 32
gradient_accumulation_steps = 1  # 있으면 포함
num_gpus = 1  # 멀티 GPU 시 사용

# 에폭 단위 로그를 위한 logging_steps 계산
logging_steps = (num_training_samples // (per_device_train_batch_size * gradient_accumulation_steps * num_gpus))


# 학습 설정
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=logging_steps,
    metric_for_best_model="mse",  # ✅ compute_metrics에 꼭 있어야 함
    greater_is_better=False,
    fp16=torch.cuda.is_available(),  # ✅ 안전함
)

# ✅ Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,  # 실험용으로 train=eval
    compute_metrics=compute_metrics,
)

# ✅ 학습 시작
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hyun_0415 (hyun_0415-kyung-hee-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Mae,Mse,Rmse,Mape
1,1.615500,0.325990,0.351256,0.325990,0.570956,10.399855
2,0.293700,0.320399,0.350102,0.320399,0.566038,10.496989
3,0.261600,0.303276,0.329739,0.303276,0.550705,9.704084
4,0.235500,0.327373,0.352351,0.327373,0.572166,10.268517
5,0.216100,0.298913,0.338285,0.298913,0.546730,9.675474


TrainOutput(global_step=7300, training_loss=0.5243023583007186, metrics={'train_runtime': 2544.006, 'train_samples_per_second': 91.788, 'train_steps_per_second': 2.869, 'total_flos': 0.0, 'train_loss': 0.5243023583007186, 'epoch': 5.0})

# Train

In [22]:
# 학습 설정
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=100,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=10,
    metric_for_best_model="mse",  # ✅ compute_metrics에 꼭 있어야 함
    greater_is_better=False,
    fp16=torch.cuda.is_available(),  # ✅ 안전함
)

# ✅ Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,  # 실험용으로 train=eval
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]  # ✅ 여기에 patience 설정
)

# ✅ 학습 시작
trainer.train()

Epoch,Training Loss,Validation Loss,Mae,Mse,Rmse,Mape
1,0.201300,0.308277,0.338128,0.308277,0.555227,9.506412
2,0.135800,0.345892,0.380278,0.345892,0.588125,10.924996
3,0.187900,0.386959,0.409331,0.386959,0.622060,11.523282
4,0.132300,0.321792,0.363377,0.321792,0.567267,10.004059
5,0.129200,0.345100,0.371939,0.345100,0.587452,10.641022
6,0.128700,0.350602,0.391356,0.350602,0.592117,10.942223


TrainOutput(global_step=8760, training_loss=0.15510694888084448, metrics={'train_runtime': 2995.6485, 'train_samples_per_second': 1558.995, 'train_steps_per_second': 48.737, 'total_flos': 0.0, 'train_loss': 0.15510694888084448, 'epoch': 6.0})

In [23]:
# 평가
results = trainer.evaluate()

# 평가 결과 출력
print("Evaluation Results:")
for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

Evaluation Results:
eval_loss: 0.3083
eval_mae: 0.3381
eval_mse: 0.3083
eval_rmse: 0.5552
eval_mape: 9.5064
eval_runtime: 29.8194
eval_samples_per_second: 223.7470
eval_steps_per_second: 3.5210
epoch: 6.0000


In [ ]:
# pd.set_option('display.max_colwidth', None)

# error_sample = df.loc[df['LLaMA_Output'].str.contains('Input')==True, ['title_y', 'text', 'LLaMA_Output']].tail(10)


In [ ]:
import numpy as np

# 성능 데이터 입력
results = {
    "Segment": {
        "MAE":  [0.3781, 0.3927, 0.3824, 0.3723, 0.3700],
        "MSE":  [0.3322, 0.3293, 0.3155, 0.3259, 0.3147],
        "RMSE": [0.5764, 0.5739, 0.5617, 0.5709, 0.5610],
        "MAPE": [0.1048, 0.1064, 0.1059, 0.1046, 0.1032],
    }
}

# 평균과 표준편차 계산
summary = {}

for model, metrics in results.items():
    summary[model] = {}
    for metric, values in metrics.items():
        values_arr = np.array(values)
        summary[model][metric] = {
            "mean": round(values_arr.mean(), 6),
            "std": round(values_arr.std(ddof=1), 6)   # 표본 표준편차
        }

# 결과 출력
for model, metrics in summary.items():
    print(f"Model: {model}")
    for metric, stats in metrics.items():
        print(f"  {metric} -> Mean: {stats['mean']}, Std: {stats['std']}")


Model: Segment
  MAE -> Mean: 0.3791, Std: 0.009026
  MSE -> Mean: 0.32352, Std: 0.008008
  RMSE -> Mean: 0.56878, Std: 0.007061
  MAPE -> Mean: 0.10498, Std: 0.001246
